In [29]:
from config import settings
import json
from datetime import datetime, timezone, timedelta
import requests
from bank_client import jwt_gen
import hashlib

import os
os.chdir("..")

In [ ]:
API_ORIGIN = "https://api.enablebanking.com"
ASPSP_NAME = "Revolut"
ASPSP_COUNTRY = "HU"

with open(settings.sessions_info_path, "r") as f:
    session_info = json.load(f)

base_headers = jwt_gen()

# Using the first available account for the following API calls
account_uid = session_info["Revolut"]['accounts'][0]['uid']

all_transactions = []

# Retrieving account balances
r = requests.get(f"{API_ORIGIN}/accounts/{account_uid}/balances", headers=base_headers)
if r.status_code == 200:
    print("Balances:")
    print(r.json())
else:
    print(f"Error response {r.status_code}:", r.text)


# Retrieving account transactions (since 90 days ago)
query = {
    "date_from": (datetime.now(timezone.utc) - timedelta(days=90)).date().isoformat(),
}
continuation_key = None
while True:
    if continuation_key:
        query["continuation_key"] = continuation_key
    r = requests.get(
        f"{API_ORIGIN}/accounts/{account_uid}/transactions",
        params=query,
        headers=base_headers,
    )
    if r.status_code == 200:
        resp_data = r.json()
        all_transactions.extend(resp_data["balances"])
        continuation_key = resp_data.get("continuation_key")
        if not continuation_key:
            print("No continuation key. All transactions were fetched")
            break
        print(f"Going to fetch more transactions with continuation key {continuation_key}")
    else:
        print(f"Error response {r.status_code}:", r.text)

with open("./data/transactions_revolut.json", "w") as f:
    json.dump(all_transactions, f, indent=2)

print("All done!")

Balances:
{'balances': [{'name': 'Available balance calculated in the course of the business day', 'balance_amount': {'currency': 'HUF', 'amount': '115959.85'}, 'balance_type': 'ITAV', 'last_change_date_time': None, 'reference_date': '2026-06-19', 'last_committed_transaction': None}]}
No continuation key. All transactions were fetched
All done!


In [5]:
settings.BANKS[0]["name"]

'Erste Bank'

In [1]:
from decimal import Decimal

In [4]:
test = "20.00"

testdec = Decimal(test.strip(' "'))

In [15]:
import json

with open("../data/transactions_revolut_huf.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [18]:
for tr in data:
    if len(tr["remittance_information"]) > 1:
        print(tr["remittance_information"])

['2 months of deposit + first half month of rent, common cost and internet', 'To Yusuke F']
['Kaucio(310k)+ junius felhavi berleti díj', 'From Lili P']
['🍕', 'To REGINA BUNEVACZ']
['Sent from Revolut ', 'To Gábor Soós']
['Gabor horvatorszag - koszi!', 'To Márton P']
['Skacos este', 'From Lili P']
['Szpot', 'From Lili P']
['Kobalt', 'From Lili P']


In [21]:
with open("../secrets/sessions.json", "r", encoding="utf-8") as f:
    sessions = json.load(f)

In [22]:
sessions

{'Erste Bank': {'access': {'accounts': None,
   'balances': True,
   'transactions': True,
   'valid_until': '2026-11-15T11:20:26.484107Z'},
  'accounts': [{'account_id': {'iban': 'HU68116000060000000197332401',
     'other': None},
    'account_servicer': {'bic_fi': 'GIBAHUH0',
     'clearing_system_member_id': None,
     'name': None},
    'all_account_ids': [{'identification': 'HU68116000060000000197332401',
      'issuer': 'HU',
      'scheme_name': 'IBAN'},
     {'identification': '116000060000000197332401',
      'issuer': 'HU',
      'scheme_name': 'BBAN'}],
    'cash_account_type': 'CACC',
    'credit_limit': None,
    'currency': 'HUF',
    'details': None,
    'identification_hash': 'WwpbCiJhY2NvdW50IiwKImFjY291bnRfaWQiLAoiaWJhbiIKXSwKWwoiYWNjb3VudCIsCiJjdXJyZW5jeSIKXQpd.NOpe0LQ05gg4rjdk6Px2ENlo5H5272DGd/PwClTxzcY=',
    'identification_hashes': ['WwpbCiJhY2NvdW50IiwKImFjY291bnRfaWQiLAoiaWJhbiIKXQpd.+X9JoQ7JFoqKKNgwx22dXzvScVBUqockohGiKH3EBjM=',
     'WwpbCiJhY2NvdW50IiwKImFj

In [31]:
with open("data/transactions_revolut_huf.json") as f:
    data = json.load(f)

bank_name = "Revolut"
account_uid = "0c843455-1b77-47e3-8856-06df2118fc1d"

for i, tr in enumerate(data):
    entry_ref = tr.get("entry_reference") or "NO_REF"
    unique_id = hashlib.sha256(
        f"{bank_name}_{account_uid}_{entry_ref}".encode()
    ).hexdigest()
    print(f"{i}: {entry_ref} -> {unique_id[:8]}...")

0: 6a335458-38da-aaa6-baad-281eb8b82a40 -> 8fae6b39...
1: 6a335458-fbaf-a111-921e-c24141f07bb1 -> 00d874d5...
2: 6a331234-b071-a425-9289-fe8dae228f40 -> 2a7c4add...
3: 6a331234-fe56-ad3a-bdee-d5905b15863f -> 6ea6a3cf...
4: 6a32f208-a5cd-a11c-b3de-c7db3177d013 -> 88c4c4b3...
5: 6a32f207-fa7c-a067-b8ce-d4fcd4c1c5f7 -> dfbc16a6...
6: 6a31848a-a8e0-acb4-b6a4-e90d9fad51d9 -> c6ac621c...
7: 6a31848a-2027-a225-9e63-a269704337b5 -> a198f099...
8: 6a316224-cef5-ab4d-a7df-f5b95ed3acf1 -> 16e95130...
9: 6a316224-0d9f-a415-8708-deb435466847 -> 4c1e7ed3...
10: 6a31620e-22ff-a1a4-9eb2-6a99efc3f91f -> 713395c0...
11: 6a316138-8cb6-a2bc-82f3-ad65c41b81e2 -> 18d35896...
12: 6a316138-e5f2-a9f8-8006-ebfc149a794a -> 752b55f9...
13: 6a314f70-be29-aac6-bd8d-d896e3d971da -> 79a9535f...
14: 6a314f70-d814-af00-a6ec-f84884c96649 -> 6272ec5f...
15: 6a314f58-fca1-a14a-89f8-d6b83312901f -> 048191ae...
16: 6a314f3c-494c-a208-b98b-e38c7e0d0c27 -> 8dd0ba17...
17: 6a3080d8-092c-a197-9c0c-eb705a309257 -> 89ad8b5e...
18

In [32]:
import json
from collections import Counter

with open("data/transactions_revolut_huf.json") as f:
    transactions = json.load(f)

refs = [tr.get("entry_reference") for tr in transactions]
counts = Counter(refs)
duplicates = {ref: count for ref, count in counts.items() if count > 1}
print(duplicates)

{}


In [33]:
refs = [tr.get("entry_reference") for tr in transactions]
print([r for r in refs if refs.count(r) > 1])

[]


In [35]:
import hashlib
from collections import Counter

bank_name = "Revolut"
account_uid = "0c843455-1b77-47e3-8856-06df2118fc1d"

hashes = []
for i, tr in enumerate(transactions):
    entry_ref = tr.get("entry_reference") or "NO_REF"
    unique_id = hashlib.sha256(
        f"{bank_name}_{account_uid}_{entry_ref}".encode()
    ).hexdigest()
    hashes.append((i, unique_id, entry_ref))

hash_counts = Counter(h[1] for h in hashes)
duplicates = [h for h in hashes if hash_counts[h[1]] > 1]
for d in duplicates:
    print(f"Index {d[0]}: ref={d[2]}, hash={d[1][:16]}...")

In [37]:
API_ORIGIN = "https://api.enablebanking.com"

r = requests.get(f"{API_ORIGIN}/accounts/{account_uid}/balances", headers=base_headers)
print(r.json())

NameError: name 'base_headers' is not defined

In [40]:
len(session_info["Revolut"]["accounts"])

3